In [49]:
import pandas as pd
import numpy as np
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import csr_matrix

In [50]:
# Cleaning events dataset
events = pd.read_csv("/content/events.csv",dtype=str)
events.drop(columns=['city','state', 'zip', 'country','lat','lng','user_id','start_time'], inplace=True)
events.rename(columns={'event_id': 'event'}, inplace=True)

In [51]:
events.head()

,event,c_1,c_2,c_3,c_4,c_5,c_6,c_7,c_8,c_9,...,c_92,c_93,c_94,c_95,c_96,c_97,c_98,c_99,c_100,c_other
0,684921758,2,0,2,0,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,9
1,244999119,2,0,2,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,7
2,3928440935,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,12
3,2582345152,1,0,2,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,8
4,1051165850,1,1,0,0,0,0,0,2,0,...,0,0,0,0,0,0,0,0,0,9


In [52]:
events.shape

(1029589, 102)

In [53]:
# Cleaning train dataset
train = pd.read_csv('/content/train.csv',dtype=str)
train = train.drop(columns=['timestamp','invited'])
train['interested'] = train['interested'].astype(int)
train['not_interested'] = train['not_interested'].astype(int)

In [54]:
train.head()

,user,event,interested,not_interested
0,3044012,1918771225,0,0
1,3044012,1502284248,0,0
2,3044012,2529072432,1,0
3,3044012,3072478280,0,0
4,3044012,1390707377,0,0


In [55]:
train.shape

(15398, 4)

In [56]:
# Merging train and event
train_events=train.merge(events, on='event')

In [167]:
train_events.head()

,user,event,interested,not_interested,c_1,c_2,c_3,c_4,c_5,c_6,...,c_93,c_94,c_95,c_96,c_97,c_98,c_99,c_100,c_other,response
0,3044012,2529072432,1,0,2,0,0,0,0,0,...,0,0,1,0,0,0,0,0,37,1
1,23170479,2529072432,1,0,2,0,0,0,0,0,...,0,0,1,0,0,0,0,0,37,1
2,26389537,2529072432,0,0,2,0,0,0,0,0,...,0,0,1,0,0,0,0,0,37,0
3,38974975,2529072432,1,0,2,0,0,0,0,0,...,0,0,1,0,0,0,0,0,37,1
4,70152834,2529072432,1,0,2,0,0,0,0,0,...,0,0,1,0,0,0,0,0,37,1


In [58]:
# Merging interested and not_interested of train dataset to one colunm response that has 3 values 1 means interested, 0 no response, -1 not interested
def merge_columns(row):
    if row['interested'] == 1:
        return 1
    elif row['not_interested'] == 1:
        return -1
    else:
        return 0

train_events['response'] = train_events.apply(merge_columns, axis=1)

In [59]:
unique_values=train_events['response'].unique()
print(unique_values)

[ 1  0 -1]


In [60]:
train_events.shape

(5531, 106)

In [61]:
# creating dataframe for doing KNN to find similar users based on their response
transformed_df = train_events.pivot_table(index='user', columns='event', values='response',fill_value=0)
transformed_df = transformed_df.reset_index()

In [62]:
transformed_df.shape

(1933, 3207)

In [63]:
transformed_df.head()

event,user,1000481836,1001013482,1002636124,1002911800,1005830738,1006903887,1009755933,1010067902,101008559,...,98657663,988131255,992238642,992455200,992561323,994276872,996491337,996645840,996787136,999643248
0,1000293064,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,1006838695,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,1008893291,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,1011223331,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,1016040879,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [64]:
unique_values = transformed_df.iloc[0].unique()
print(unique_values)

['1000293064' 0]


In [65]:
# KNN model with 10 neighbour to find similar users
transformed_df=transformed_df.astype(np.int64)
event_sparse = csr_matrix(transformed_df)
model = NearestNeighbors(algorithm = 'brute')
model.fit(event_sparse)

NearestNeighbors(algorithm='brute')

In [66]:
distance, suggestion = model.kneighbors(transformed_df, n_neighbors=10)
user_list = transformed_df['user']

/usr/local/lib/python3.10/dist-packages/sklearn/base.py:432: UserWarning: X has feature names, but NearestNeighbors was fitted without feature names
  warnings.warn(


In [67]:
# create key: user , value: similar users
similar_users_table = pd.DataFrame()
similar_users_table['User'] = user_list
for i in range(10):
    similar_users_table[f'Similar_User_{i+1}'] = transformed_df.iloc[suggestion[:, i], 0].values

In [68]:
similar_users_table.shape

(1933, 11)

In [69]:
similar_users_table.head()

,User,Similar_User_1,Similar_User_2,Similar_User_3,Similar_User_4,Similar_User_5,Similar_User_6,Similar_User_7,Similar_User_8,Similar_User_9,Similar_User_10
0,1000293064,1000293064,996222490,995603012,1006838695,1008893291,990496397,990116823,1011223331,988213942,988160405
1,1006838695,1006838695,1008893291,1011223331,1000293064,1016040879,1016099896,1017178142,996222490,1017536864,995603012
2,1008893291,1008893291,1006838695,1011223331,1016040879,1016099896,1017178142,1000293064,1017536864,1018886228,996222490
3,1011223331,1011223331,1008893291,1006838695,1016040879,1016099896,1017178142,1017536864,1018886228,1000293064,1023717643
4,1016040879,1016040879,1016099896,1017178142,1017536864,1018886228,1011223331,1008893291,1023717643,1006838695,1025593498


In [78]:
# redusing dataset of event based on the event list in user dataset
column_names = transformed_df.columns[1:]
events_filtered = events[events['event'].isin(column_names)]
events_filtered.shape

(3206, 102)

In [79]:
events_filtered.head()

,event,c_1,c_2,c_3,c_4,c_5,c_6,c_7,c_8,c_9,...,c_92,c_93,c_94,c_95,c_96,c_97,c_98,c_99,c_100,c_other
2,3928440935,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,12
3,2582345152,1,0,2,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,8
5,1212611096,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,22
6,3689283674,0,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,28
7,2584113432,0,0,2,0,0,33,0,3,1,...,2,0,0,0,0,0,0,0,0,354


In [128]:
# finding similarity for events based on cosine similarity
feature_columns = [f'c_{i}' for i in range(1, 101)]
events_final = events_filtered[feature_columns]
events_final.apply(pd.to_numeric, errors='coerce')
event_similarity = cosine_similarity(events_final)
event_similarity

array([[1.        , 0.        , 0.        , ..., 0.        , 0.70710678,
        0.        ],
       [0.        , 1.        , 0.        , ..., 0.26958193, 0.        ,
        0.35147975],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       ...,
       [0.        , 0.26958193, 0.        , ..., 1.        , 0.        ,
        0.54810729],
       [0.70710678, 0.        , 0.        , ..., 0.        , 1.        ,
        0.        ],
       [0.        , 0.35147975, 0.        , ..., 0.54810729, 0.        ,
        1.        ]])

In [130]:
# Building similar_events_table that contains key: event , value: list of similar events
print(events_filtered.shape)
similar_events_table = pd.DataFrame(index=events_filtered['event'],columns=range(50))
n, _ = event_similarity.shape
print(n)
for i in range(n):
  count = 0
  for j in range(n):
    if(event_similarity[i][j] > 0.5):
      similar_events_table.iloc[i, count] = events_filtered.iloc[j,0]
      count+=1
    if(count>=50):
      break
similar_events_table = similar_events_table.fillna('')
similar_events_table.head()

(3206, 102)
3206


,0,1,2,3,4,5,6,7,8,9,...,40,41,42,43,44,45,46,47,48,49
event,,,,,,,,,,,,,,,,,,,,,
3928440935,3928440935,122777568,3949287071,788496445,3186605940,2912281746,4014449816,2452750556,1645370294,4249581931,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2582345152,2582345152,298169907,1820269907,1929622843,3980763324,297856962,4004496621,739705932,2998372996,920467258,...,44776435,1859823732,42204521,3385943756,706200405,2042113559,1386404286,3475854859,3531332626,3773489407
1212611096,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3689283674,3689283674,1609864127,3028497756,3790866767,585272047,4092423355,3676617790,2997075790,4222144697,88377811,...,2772929791,1638625563,3949816728,3173523505,1669979138,803811392,2204013412,1640004842,2773966086,2489574463
2584113432,2584113432,2773204108,2622371373,920600431,3212079634,3618436742,2915353594,3349548561,1801869689,3213836603,...,2027514460,2652418462,4002262960,1279926604,2742780288,3547892636,4259189014,2006849293,4281032788,26815716


In [92]:
similar_events_table.shape

(3206, 0)

In [166]:
# Building final recomendation for each users in train by merging user content based similarity and user based similarity
recommendation_based_for_user =  pd.DataFrame(index=user_list,columns=range(100))
event_list = list(transformed_df.columns)
for i in suggestion:
  similar_events = set()
  for j in i:
    z  = 0
    for k in transformed_df.iloc[j]:
      z+=1
      if(k == 1):
        similar_events.add(event_list[z])
  final_event_rec = set()
  for e in similar_events:
    for eve in (similar_events_table[similar_events_table.index==e].values[0]):
      final_event_rec.add(eve)
  c = 0
  for eve in final_event_rec:
    recommendation_based_for_user.iloc[i[0], c] = eve
    c +=1
    if(c>=100):
      break
recommendation_based_for_user.head()

,0,1,2,3,4,5,6,7,8,9,...,90,91,92,93,94,95,96,97,98,99
user,,,,,,,,,,,,,,,,,,,,,
1000293064,2622371373,3088405938,2821117540,2938749256,539032065,2424074793,2129139974,2404440151,3247562562,298169907,...,3418014649,2491721612,4032767837,1455527953,572112673,4106103231,1397608202,2573061069,1310511988,4281032788
1006838695,2186630028,2622371373,3088405938,764214719,2821117540,1862523963,2938749256,1771107019,539032065,2424074793,...,2127623176,4216192606,1711893150,3451292933,2422810156,2498438372,4023489552,175933138,920467258,4166625784
1008893291,3929337192,2821117540,2938749256,2404440151,301530247,1569036874,2600467156,124311055,4002262960,4092423355,...,1929622843,1639958590,2042697619,677713566,2533309120,1959421009,3618436742,1769487166,2731002498,3080870370
1011223331,3929337192,2821117540,2938749256,2404440151,301530247,1569036874,2600467156,124311055,4002262960,3374746454,...,3088405938,1771107019,3801433102,861702567,3331186835,524691289,3678607925,1100725936,1783910332,4276436069
1016040879,3929337192,2821117540,2938749256,2404440151,301530247,1569036874,2600467156,124311055,4002262960,3374746454,...,3088405938,1771107019,3801433102,861702567,3331186835,524691289,3678607925,1100725936,1783910332,4276436069


In [164]:
recommendation_based_for_user.shape

(1933, 100)

In [165]:
# Print the number of unique values for each column
unique_counts = recommendation_based_for_user.nunique()
print(unique_counts)

0      23
1      41
2      63
3      69
4      74
     ... 
95    405
96    397
97    408
98    407
99    399
Length: 100, dtype: int64
